# 🚗 Driver Risk Segmentation for Usage-Based Insurance

> **Business context** — Traditional car insurance pricing relies on static demographic variables (age, gender, vehicle type) that are poor proxies for actual driving risk. Telematics technology now makes it possible to collect real-time behavioral data from drivers. This project leverages **unsupervised machine learning (clustering)** to segment drivers into meaningful risk profiles, enabling insurers to offer **personalized, behavior-based premiums**.

---

## 🎯 Business Objective

**Problem statement:** An insurance company wants to move from demographic-based pricing to **usage-based insurance (UBI)**. The goal is to identify homogeneous driver profiles based on actual driving behavior so that:

- 🟢 **Safe drivers** are rewarded with lower premiums
- 🔴 **High-risk drivers** are correctly priced to reflect their risk exposure
- 📊 The insurer improves **portfolio risk management** and reduces claims frequency

**Approach:** Since we have no pre-labeled risk categories, this is an **unsupervised learning** problem. We apply K-Means clustering on telematics-derived features to discover natural driver segments.

---

## 📁 Notebook Structure

| Step | Section |
|------|---------|
| 1 | Data Loading & First Look |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Data Preprocessing |
| 4 | Optimal Number of Clusters (Elbow + Silhouette) |
| 5 | K-Means Clustering |
| 6 | Cluster Profiling & Business Interpretation |
| 7 | Actionable Recommendations |

## 1. Setup & Data Loading

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Scikit-learn ────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

# ── Settings ────────────────────────────────────────────────────────────────
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42

print('✅ All libraries loaded successfully.')

In [ ]:
# Load the dataset
df = pd.read_csv('../data/driver_data.csv')

print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(10)

## 2. Exploratory Data Analysis (EDA)

Before any modelling, we need to understand the data — its structure, distributions, and potential issues. EDA is not just a formality: it directly informs preprocessing choices and helps us validate that the features are meaningful for the business problem.

In [ ]:
# ── Data types and missing values ────────────────────────────────────────────
print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Missing Values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values found. ✅')

In [ ]:
# ── Descriptive statistics ───────────────────────────────────────────────────
# Key insight: look at the range and spread of each feature.
# Large differences in scale between features will require normalisation before clustering.
df.describe()

In [ ]:
# ── Feature distributions ────────────────────────────────────────────────────
# We visualise each feature independently to detect skewness and outliers.
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

n_cols = 3
n_rows = int(np.ceil(len(numeric_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ──────────────────────────────────────────────────────
# Highly correlated features carry redundant information.
# While K-Means is not a regression model, redundant features can distort distance calculations
# and give disproportionate weight to correlated dimensions.
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))  # show lower triangle only
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    ax=ax, cbar_kws={'shrink': 0.8}
)
ax.set_title('Feature Correlation Matrix', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Outlier detection via boxplots ───────────────────────────────────────────
# Outliers can significantly distort cluster centroids in K-Means.
# We identify them visually before deciding on a treatment strategy.

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightsteelblue', color='steelblue'),
                    medianprops=dict(color='crimson', linewidth=2))
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Value')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Outlier Detection — Boxplots by Feature', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

K-Means relies on **Euclidean distances** to assign points to clusters. This makes it highly sensitive to feature scale. A feature measured in thousands (e.g. kilometres driven) would dominate a feature measured in units (e.g. number of harsh brakes), even if both are equally informative.

**→ Standardisation (z-score scaling) is mandatory before applying K-Means.**

In [ ]:
# ── Select features for clustering ──────────────────────────────────────────
# We retain all numeric features. Adjust this list based on domain knowledge
# or correlation analysis above (drop highly redundant features if needed).
features = numeric_cols.copy()

X = df[features].copy()
print(f'Features used for clustering ({len(features)}): {features}')

In [ ]:
# ── Handle missing values ────────────────────────────────────────────────────
# We use median imputation — robust to outliers and appropriate for skewed distributions.
if X.isnull().any().any():
    X.fillna(X.median(), inplace=True)
    print('Missing values imputed with column medians.')
else:
    print('No missing values — no imputation needed. ✅')

In [ ]:
# ── Standardise features ─────────────────────────────────────────────────────
# StandardScaler transforms each feature to mean=0, std=1.
# This ensures all features contribute equally to the distance metric.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=features)

print('After standardisation:')
print(X_scaled_df.describe().loc[['mean', 'std']].round(3))

## 4. Finding the Optimal Number of Clusters

K-Means requires us to specify `k` (the number of clusters) upfront. Rather than guessing arbitrarily, we use two complementary methods:

- **Elbow Method**: plots inertia (within-cluster sum of squares) vs. k. The "elbow" — where the rate of decrease sharply slows — indicates the point of diminishing returns.
- **Silhouette Analysis**: measures how similar a point is to its own cluster versus neighbouring clusters. A higher silhouette score (max = 1) indicates better-defined clusters.

In [ ]:
# ── Elbow Method ─────────────────────────────────────────────────────────────
k_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init='auto')
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
axes[0].plot(list(k_range), inertias, marker='o', color='steelblue', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of clusters (k)', fontsize=12)
axes[0].set_ylabel('Inertia (WCSS)', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=14, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.6)

# Silhouette plot
axes[1].plot(list(k_range), silhouette_scores, marker='s', color='crimson', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of clusters (k)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Analysis', fontsize=14, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.suptitle('Determining the Optimal Number of Clusters', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

best_k_sil = list(k_range)[np.argmax(silhouette_scores)]
print(f'\nBest k by silhouette score: k = {best_k_sil} (score = {max(silhouette_scores):.3f})')

## 5. K-Means Clustering

We apply K-Means with the optimal `k` identified above. We also run **PCA (Principal Component Analysis)** to project the high-dimensional data into 2D for visualisation — this is purely for display purposes and does not affect the clustering.

In [ ]:
# ── Fit final K-Means model ──────────────────────────────────────────────────
# We use the k suggested by silhouette analysis.
# You can override this if the elbow plot suggests a different value.
OPTIMAL_K = best_k_sil

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=RANDOM_STATE, n_init='auto')
df['Cluster'] = kmeans.fit_predict(X_scaled)

print(f'K-Means fitted with k = {OPTIMAL_K}')
print(f'Final inertia: {kmeans.inertia_:.2f}')
print(f'Final silhouette score: {silhouette_score(X_scaled, df["Cluster"]):.3f}')
print()
print('Driver count per cluster:')
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# ── PCA for 2D visualisation ─────────────────────────────────────────────────
# PCA reduces dimensionality while preserving maximum variance.
# The two principal components let us visualise cluster separability.
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = df['Cluster'].values

explained = pca.explained_variance_ratio_
print(f'PC1 explains {explained[0]*100:.1f}% of variance')
print(f'PC2 explains {explained[1]*100:.1f}% of variance')
print(f'Total explained: {sum(explained)*100:.1f}%')

In [ ]:
# ── Cluster scatter plot ─────────────────────────────────────────────────────
PALETTE = sns.color_palette('tab10', OPTIMAL_K)

fig, ax = plt.subplots(figsize=(10, 7))

for cluster_id in range(OPTIMAL_K):
    mask = pca_df['Cluster'] == cluster_id
    ax.scatter(
        pca_df.loc[mask, 'PC1'],
        pca_df.loc[mask, 'PC2'],
        label=f'Cluster {cluster_id}',
        color=PALETTE[cluster_id],
        alpha=0.65, s=50, edgecolors='white', linewidths=0.4
    )

# Plot centroids projected onto PCA space
centroids_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    marker='X', s=250, color='black', zorder=10, label='Centroids'
)

ax.set_xlabel(f'Principal Component 1 ({explained[0]*100:.1f}% variance)', fontsize=12)
ax.set_ylabel(f'Principal Component 2 ({explained[1]*100:.1f}% variance)', fontsize=12)
ax.set_title('Driver Clusters — PCA 2D Projection', fontsize=15, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
# ── Silhouette plot per sample ───────────────────────────────────────────────
# This plot shows how well each individual driver fits its assigned cluster.
# Wide, positive bars = well-assigned. Negative bars = likely misclassified.

sample_silhouette_values = silhouette_samples(X_scaled, df['Cluster'])

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10

for i in range(OPTIMAL_K):
    ith_silhouette_values = sample_silhouette_values[df['Cluster'] == i]
    ith_silhouette_values.sort()
    size_cluster_i = ith_silhouette_values.shape[0]
    y_upper = y_lower + size_cluster_i

    ax.fill_betweenx(
        np.arange(y_lower, y_upper),
        0, ith_silhouette_values,
        facecolor=PALETTE[i], alpha=0.8
    )
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i), fontsize=11, fontweight='bold')
    y_lower = y_upper + 10

avg_score = silhouette_score(X_scaled, df['Cluster'])
ax.axvline(x=avg_score, color='crimson', linestyle='--', linewidth=2, label=f'Avg score = {avg_score:.3f}')
ax.set_xlabel('Silhouette Coefficient', fontsize=12)
ax.set_ylabel('Cluster', fontsize=12)
ax.set_title('Silhouette Plot — Individual Driver Scores', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

## 6. Cluster Profiling & Business Interpretation

Clustering is only useful if the resulting segments are **interpretable and actionable**. Here we characterise each cluster by computing the mean of each feature per cluster, then translate those statistics into meaningful driver profiles.

In [ ]:
# ── Cluster means on original (non-scaled) features ──────────────────────────
cluster_profile = df.groupby('Cluster')[features].mean().round(2)
cluster_profile.index.name = 'Cluster'
cluster_profile

In [ ]:
# ── Heatmap of standardised cluster centroids ─────────────────────────────────
# Using standardised values makes it easy to compare across features with different units.
centroids_df = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=features,
    index=[f'Cluster {i}' for i in range(OPTIMAL_K)]
)

fig, ax = plt.subplots(figsize=(max(10, len(features) * 1.2), OPTIMAL_K + 2))
sns.heatmap(
    centroids_df, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, linewidths=0.5,
    ax=ax, cbar_kws={'label': 'Standardised value (z-score)'}
)
ax.set_title('Cluster Centroids — Standardised Feature Values\n(Green = above average  |  Red = below average)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Feature', fontsize=11)
ax.set_ylabel('Cluster', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature distribution per cluster (violin plots) ──────────────────────────
# Violin plots show the full distribution shape (not just the mean),
# revealing whether clusters truly separate on each feature.

n_features = len(features)
n_cols = 3
n_rows = int(np.ceil(n_features / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(features):
    sns.violinplot(
        data=df, x='Cluster', y=col,
        palette=PALETTE[:OPTIMAL_K],
        inner='box', ax=axes[i]
    )
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('Cluster')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions per Cluster', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Radar chart — cluster risk fingerprints ───────────────────────────────────
# A radar chart gives an at-a-glance "fingerprint" of each cluster
# on all dimensions simultaneously — ideal for business presentations.

from matplotlib.patches import FancyArrowPatch

# Normalise centroids to [0, 1] for the radar
centroids_norm = (centroids_df - centroids_df.min()) / (centroids_df.max() - centroids_df.min() + 1e-9)

categories = list(centroids_norm.columns)
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]  # close the loop

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

for idx, row in centroids_norm.iterrows():
    values = row.tolist()
    values += values[:1]
    cluster_id = int(idx.split()[-1])
    ax.plot(angles, values, linewidth=2, color=PALETTE[cluster_id], label=idx)
    ax.fill(angles, values, color=PALETTE[cluster_id], alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=10)
ax.set_yticklabels([])
ax.set_title('Driver Risk Fingerprints by Cluster', fontsize=14, fontweight='bold', pad=25)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=11)

plt.tight_layout()
plt.show()

## 7. Actionable Business Recommendations

The table below translates each cluster's statistical profile into a **business label and concrete insurance action**. These interpretations are derived from the cluster centroids and should be validated with domain experts before implementation.

In [ ]:
# ── Summary table: cluster → business profile → recommendation ───────────────
# NOTE: The labels below are illustrative. Adjust them based on the
# actual centroid values observed in your dataset.

cluster_labels = {
    # Update these based on your cluster profiling results above
    0: ('🟢 Safe & Experienced Driver',
        'Low speed variance, few incidents, high mileage consistency.',
        'Offer loyalty discount (−10% to −20%). Flag as low-risk tier.'),
    1: ('🟡 Moderate Risk — Urban Commuter',
        'High frequency of short trips, frequent braking, moderate incidents.',
        'Standard premium. Offer telematics-based coaching to reduce risk score.'),
    2: ('🔴 High-Risk Driver',
        'High speed events, frequent harsh acceleration/braking, elevated incident rate.',
        'Apply risk surcharge (+15% to +30%). Mandatory quarterly review.'),
    3: ('🔵 Occasional & Prudent Driver',
        'Very low mileage, calm driving style, long intervals between trips.',
        'Offer pay-per-mile (PPM) product — lower base premium with mileage cap.'),
}

rows = []
for cluster_id, (label, profile, action) in cluster_labels.items():
    if cluster_id < OPTIMAL_K:
        count = (df['Cluster'] == cluster_id).sum()
        pct = count / len(df) * 100
        rows.append({
            'Cluster': cluster_id,
            'Business Label': label,
            'Driving Profile': profile,
            'Count': count,
            '% of Fleet': f'{pct:.1f}%',
            'Recommended Action': action
        })

recommendations = pd.DataFrame(rows).set_index('Cluster')

with pd.option_context('display.max_colwidth', None):
    display(recommendations)

In [ ]:
# ── Fleet risk composition chart ──────────────────────────────────────────────
cluster_sizes = df['Cluster'].value_counts().sort_index()
labels_short = [cluster_labels[i][0] if i in cluster_labels else f'Cluster {i}'
                for i in cluster_sizes.index]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(
    labels_short, cluster_sizes.values,
    color=[PALETTE[i] for i in cluster_sizes.index],
    edgecolor='white', height=0.6
)

for bar, count in zip(bars, cluster_sizes.values):
    pct = count / len(df) * 100
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
            f'{count:,} ({pct:.1f}%)', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Number of Drivers', fontsize=12)
ax.set_title('Fleet Composition by Risk Segment', fontsize=14, fontweight='bold')
ax.grid(True, axis='x', linestyle='--', alpha=0.5)
ax.set_xlim(0, cluster_sizes.max() * 1.2)
plt.tight_layout()
plt.show()

In [ ]:
# ── Export segmented dataset ─────────────────────────────────────────────────
# Save the original dataset enriched with cluster labels for downstream use
# (CRM integration, actuarial modelling, BI dashboards, etc.)
output_path = '../data/driver_data_segmented.csv'
df.to_csv(output_path, index=False)
print(f'✅ Segmented dataset saved to: {output_path}')
print(f'   Shape: {df.shape}')

---

## 📝 Conclusion

### What we did
We applied **K-Means clustering** on telematics-derived driving behavior data to segment drivers into distinct risk profiles. The optimal number of clusters was determined objectively using the **Elbow Method** and **Silhouette Analysis**.

### Business value
| Without segmentation | With segmentation |
|---------------------|-------------------|
| Flat pricing based on demographics | Personalised pricing based on actual behavior |
| Safe drivers subsidise risky ones | Each driver pays a fair, risk-adjusted premium |
| Reactive claims management | Proactive risk coaching for moderate-risk drivers |
| One-size-fits-all product | Tailored products (loyalty, PPM, standard, high-risk) |

### Limitations & next steps
- **Cluster stability**: run multiple seeds and use ensemble clustering to validate stability
- **Temporal dynamics**: driver behaviour changes over time — consider sliding-window re-clustering
- **Supervised validation**: if historical claims data is available, validate whether clusters correlate with actual loss ratios
- **Feature engineering**: enrich with time-of-day driving, weather conditions, road type for finer-grained profiling

---
*Notebook authored as part of a Data Mining academic project. All analysis is reproducible with the provided dataset and `requirements.txt`.*